# Project 1: How Has Life Expectancy Changed Around the World?

**Computing in Context | Columbia SIPA**

This project explores a simple question: *How has life expectancy changed over time, and how does it differ across continents?* We will use the same dataset to compute the mean, median, and mode twice: first with pandas, and then from scratch using only the Python standard library. Finally, we will make a text-based visualization without any plotting packages.

**Dataset:** Gapminder, five-year observations from 1952–2007, distributed as the [Plotly Gapminder CSV](https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv). The data contains 1,704 country-year observations. Original data project: [Gapminder](https://www.gapminder.org/data/). The CSV is included alongside this notebook so it works offline.

**Important limitation:** These observations are country-year records, not a random sample of individuals. An unweighted mean gives each country-year equal weight, regardless of population.

## 1. Load and inspect the data with pandas

We first check the columns, row count, and missing values. Our numeric variable is `lifeExp`, measured in years.

In [1]:
import pandas as pd

df = pd.read_csv("gapminder.csv")
print("Rows and columns:", df.shape)
print("Columns:", list(df.columns))
print("Missing life expectancy values:", df["lifeExp"].isna().sum())
display(df.head())

Rows and columns: (1704, 8)
Columns: ['country', 'continent', 'year', 'lifeExp', 'pop', 'gdpPercap', 'iso_alpha', 'iso_num']
Missing life expectancy values: 0


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


## 2. Mean, median, and mode with pandas

The **mean** is the arithmetic average. The **median** is the middle value after sorting. The **mode** is the most frequently occurring exact value. Since life expectancy is continuous, its exact mode is less informative than its mean or median.

In [2]:
life = df["lifeExp"].dropna()
pandas_mean = life.mean()
pandas_median = life.median()
pandas_modes = sorted(life.mode().tolist())

print(f"Mean: {pandas_mean:.3f} years")
print(f"Median: {pandas_median:.3f} years")
print("Mode(s):", pandas_modes)
print("Frequency of each mode:", int(life.value_counts().max()))

Mean: 59.474 years
Median: 60.712 years
Mode(s): [69.39]
Frequency of each mode: 4


## 3. The hard way: standard library only

This section is independent of the DataFrame above. It opens the CSV file again using `csv.DictReader`, converts the life-expectancy column to floats, and implements all three statistics manually. No pandas or `statistics` functions are used.

In [3]:
import csv

values = []
with open("gapminder.csv", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        raw = row["lifeExp"].strip()
        if raw:
            values.append(float(raw))

# Mean: sum of values divided by count, implemented with a loop.
total = 0.0
for value in values:
    total += value
hard_mean = total / len(values)

# Median: sort and select the middle observation(s).
ordered = sorted(values)
n = len(ordered)
if n % 2 == 1:
    hard_median = ordered[n // 2]
else:
    hard_median = (ordered[n // 2 - 1] + ordered[n // 2]) / 2

# Mode: count exact values using a dictionary.
counts = {}
for value in values:
    counts[value] = counts.get(value, 0) + 1
maximum_frequency = max(counts.values())
hard_modes = sorted([value for value, count in counts.items() if count == maximum_frequency])

print(f"Mean: {hard_mean:.3f} years")
print(f"Median: {hard_median:.3f} years")
print("Mode(s):", hard_modes)
print("Frequency of each mode:", maximum_frequency)

Mean: 59.474 years
Median: 60.712 years
Mode(s): [69.39]
Frequency of each mode: 4


## 4. Verify both approaches

Independent calculations should produce the same results. These checks catch mistakes in the manual implementation.

In [4]:
assert abs(pandas_mean - hard_mean) < 1e-9
assert abs(pandas_median - hard_median) < 1e-9
assert pandas_modes == hard_modes
print("All three statistics match.")

All three statistics match.


## 5. Visualization: average life expectancy by continent

For a compact comparison, each block below represents approximately **two years** of life expectancy. We calculate the averages with pandas but draw the bars using only standard Python string operations and `print()`. To make the comparison meaningful, we use the most recent year available in this dataset rather than combining all decades.

In [5]:
latest_year = int(df["year"].max())
latest = df[df["year"] == latest_year]
continent_means = latest.groupby("continent")["lifeExp"].mean().sort_values(ascending=False)

print(f"Average life expectancy by continent ({latest_year})")
print("Each █ represents approximately 2 years; values shown in years.")
print("Continent       Bar                                      Years")
for continent, average in continent_means.items():
    bar = "█" * round(average / 2)
    print(f"{continent:<15} {bar} {average:5.1f}")

Average life expectancy by continent (2007)
Each █ represents approximately 2 years; values shown in years.
Continent       Bar                                      Years
Oceania         ████████████████████████████████████████  80.7
Europe          ███████████████████████████████████████  77.6
Americas        █████████████████████████████████████  73.6
Asia            ███████████████████████████████████  70.7
Africa          ███████████████████████████  54.8


## 6. What do the results mean?

The mean and median describe the center of the country-year life-expectancy distribution. They do **not** represent the life expectancy of the average person worldwide, because a small country and a populous country receive equal weight. The mode is based on exact decimal matches, so it is sensitive to rounding and is not necessarily a meaningful “typical” life expectancy.

The chart compares continents in the final observed year. These are **unweighted averages of countries**, not population-weighted continental estimates. It provides a readable overview but does not explain why differences exist. Potential influences such as health systems, income, conflict, and infectious diseases would require additional analysis.

**Takeaway:** The same three descriptive statistics can be computed using a high-level library or basic loops, sorting, and dictionaries. The visualization then turns a set of numerical averages into a comparison that is easier to read.